<a href="https://colab.research.google.com/github/nandkishor-vasi/NLP_assign/blob/main/Assign7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import torch
import torch.nn as nn

In [23]:
import random

names = ["John", "Mary", "Alice", "Bob", "David", "Emma"]
locations = ["Paris", "London", "Mumbai", "Delhi", "Tokyo"]
orgs = ["Google", "Microsoft", "Amazon", "Infosys"]

verbs = ["works at", "lives in", "visited", "joined", "left"]
templates = [
    "{name} {verb} {org}",
    "{name} {verb} {location}",
    "{name} {verb} {org} in {location}"
]

def generate_sample():
    template = random.choice(templates)

    name = random.choice(names)
    location = random.choice(locations)
    org = random.choice(orgs)
    verb = random.choice(verbs)

    sentence = template.format(
        name=name, location=location, org=org, verb=verb
    ).split()

    labels = []
    for word in sentence:
        if word in names:
            labels.append("B-PER")
        elif word in locations:
            labels.append("B-LOC")
        elif word in orgs:
            labels.append("B-ORG")
        else:
            labels.append("O")

    return sentence, labels

# Generate large dataset
data_size = 10000
sentences = []
labels = []

for _ in range(data_size):
    s, l = generate_sample()
    sentences.append(s)
    labels.append(l)

print(sentences[0], labels[0])

['David', 'works', 'at', 'Infosys', 'in', 'Delhi'] ['B-PER', 'O', 'O', 'B-ORG', 'O', 'B-LOC']


In [24]:
word2idx = {"<PAD>": 0, "<UNK>": 1}
tag2idx = {"<PAD>": 0}

for sent in sentences:
    for word in sent:
        if word not in word2idx:
            word2idx[word] = len(word2idx)

for tag_seq in labels:
    for tag in tag_seq:
        if tag not in tag2idx:
            tag2idx[tag] = len(tag2idx)

idx2tag = {v: k for k, v in tag2idx.items()}

In [25]:
from torch.nn.utils.rnn import pad_sequence

def encode_words(sent, mapping):
    return torch.tensor([mapping.get(w, mapping["<UNK>"]) for w in sent])

def encode_tags(tags, mapping):
    return torch.tensor([mapping[t] for t in tags])

X = [encode_words(s, word2idx) for s in sentences]
y = [encode_tags(t, tag2idx) for t in labels]

X_padded = pad_sequence(X, batch_first=True, padding_value=0)
y_padded = pad_sequence(y, batch_first=True, padding_value=0)

In [26]:
class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, tagset_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.lstm = nn.LSTM(64, 128, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(256, tagset_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return self.fc(out)

In [27]:
model = BiLSTMTagger(len(word2idx), len(tag2idx))

loss_fn = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(25):
    model.train()

    optimizer.zero_grad()

    outputs = model(X_padded)

    loss = loss_fn(
        outputs.view(-1, len(tag2idx)),
        y_padded.view(-1)
    )

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 1.6039
Epoch 2, Loss: 1.5547
Epoch 3, Loss: 1.5063
Epoch 4, Loss: 1.4582
Epoch 5, Loss: 1.4102
Epoch 6, Loss: 1.3618
Epoch 7, Loss: 1.3129
Epoch 8, Loss: 1.2632
Epoch 9, Loss: 1.2128
Epoch 10, Loss: 1.1616
Epoch 11, Loss: 1.1096
Epoch 12, Loss: 1.0571
Epoch 13, Loss: 1.0041
Epoch 14, Loss: 0.9511
Epoch 15, Loss: 0.8980
Epoch 16, Loss: 0.8452
Epoch 17, Loss: 0.7927
Epoch 18, Loss: 0.7407
Epoch 19, Loss: 0.6892
Epoch 20, Loss: 0.6384
Epoch 21, Loss: 0.5885
Epoch 22, Loss: 0.5398
Epoch 23, Loss: 0.4926
Epoch 24, Loss: 0.4472
Epoch 25, Loss: 0.4041


In [28]:
def predict(sentence):
    model.eval()

    encoded = torch.tensor([
        [word2idx.get(w, word2idx["<UNK>"]) for w in sentence]
    ])

    with torch.no_grad():
        outputs = model(encoded)
        preds = torch.argmax(outputs, dim=-1)[0]

    return [idx2tag[p.item()] for p in preds]

print(predict(["Alice", "works", "at", "Google"]))

['B-PER', 'O', 'O', 'B-ORG']
